# 43 — Resource Allocation

**Engineering statement:** Resource allocation specifies adaptive serving.

Notebook 37 identified operating regimes. Notebook 43 starts the second systems arc: once the serving system knows its regime, it must allocate compute, memory, verification, latency, and reserve capacity across active requests.

The scheduler is no longer just selecting a prefix length. It is spending a constrained resource budget.


## Repository roadmap

```text
00 Context
07 Verification Resource
13 Confidence Scheduling
17 Semi-Autoregressive Design
23 Throughput Objective
29 Hardware-Aware Scheduling
37 Operating Regimes
43 Resource Allocation
49 Adaptive AI Infrastructure
53 Distributed Scheduling
```


## Start here

```text
operating regime
→ resource budget
→ allocation policy
→ verification allocation
→ batch admission
→ latency control
→ adaptive serving
```

Notebook 37 asks:

> Which operating regime is active?

Notebook 43 asks:

> Given that regime, how should resources be allocated?


In [ ]:
from pathlib import Path
import json
import zipfile
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
from IPython.display import FileLink, display

CWD = Path.cwd().resolve()
if (CWD / "figures").exists() or (CWD / "results").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "figures").exists() or (CWD.parent / "results").exists() or (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "43_resource_allocation"
FIGURES.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)

def savefig(name):
    path = FIGURES / name
    plt.tight_layout()
    plt.savefig(path, dpi=160, bbox_inches="tight")
    plt.show()
    return path

def rounded_box(ax, xy, w, h, text, fontsize=12):
    box = FancyBboxPatch(
        xy, w, h,
        boxstyle="round,pad=0.03,rounding_size=0.08",
        linewidth=1.4,
        edgecolor="black",
        facecolor="white"
    )
    ax.add_patch(box)
    ax.text(xy[0] + w/2, xy[1] + h/2, text, ha="center", va="center", fontsize=fontsize)
    return box

def arrow(ax, start, end, style="->", lw=1.8):
    ax.annotate("", xy=end, xytext=start, arrowprops=dict(arrowstyle=style, lw=lw))


## 1. Second-arc roadmap

Notebook 37 completed the first major arc: mechanisms became operating regimes. Notebook 43 begins the second arc: regimes become allocation policies.


In [ ]:
# 43_second_arc_roadmap.png
fig, ax = plt.subplots(figsize=(13.5, 5.0)); ax.axis("off"); ax.set_xlim(0,13.5); ax.set_ylim(0,5)
nodes=[("00\nContext",.4,3.1,1.25,.75),("07\nVerification\nResource",2.1,3.1,1.35,.75),("13\nConfidence\nScheduling",3.95,3.1,1.45,.75),("17\nSemi-AR\nDrafting",5.95,3.1,1.35,.75),("23\nThroughput\nObjective",7.8,3.1,1.45,.75),("29\nHardware\nConstraints",9.8,3.1,1.35,.75),("37\nOperating\nRegimes",11.55,3.1,1.25,.75)]
for label,x,y,w,h in nodes: rounded_box(ax,(x,y),w,h,label,10)
for i in range(len(nodes)-1):
    _,x,y,w,h=nodes[i]; _,x1,y1,w1,h1=nodes[i+1]; arrow(ax,(x+w+.05,y+h/2),(x1-.05,y1+h1/2))
second=[("43\nResource\nAllocation",4.1,1,1.55,.75),("49\nAdaptive\nInfrastructure",6.1,1,1.55,.75),("53\nDistributed\nScheduling",8.1,1,1.55,.75)]
for label,x,y,w,h in second: rounded_box(ax,(x,y),w,h,label,10)
for i in range(len(second)-1):
    _,x,y,w,h=second[i]; _,x1,y1,w1,h1=second[i+1]; arrow(ax,(x+w+.05,y+h/2),(x1-.05,y1+h1/2))
ax.plot([.7,12.4],[2.45,2.45], color="black", lw=1.2)
ax.text(6.6,2.58,"first arc: mechanism → operating regime",ha="center",fontsize=12)
ax.text(6.85,.35,"second arc: operating regime → resource allocation → infrastructure",ha="center",fontsize=12)
ax.set_title("Resource allocation starts the second systems arc", fontsize=20, pad=12)
savefig("43_second_arc_roadmap.png")


## 2. Resource allocation flow

Operating regimes become allocation policies. A regime is an input to a budget estimator that decides how much compute, memory, verification, latency headroom, and reserve capacity should be spent.


In [ ]:
# 43_resource_allocation_flow.png
fig, ax = plt.subplots(figsize=(12, 4.8)); ax.axis("off"); ax.set_xlim(0,12); ax.set_ylim(0,4)
nodes=[("Operating\nregime",.6,2.05,1.45,.8),("Resource\nbudget",2.8,2.05,1.45,.8),("Allocation\npolicy",5,2.05,1.45,.8),("Verification\nallocation",7.2,2.05,1.65,.8),("Serving\nthroughput",9.75,2.05,1.55,.8)]
for label,x,y,w,h in nodes: rounded_box(ax,(x,y),w,h,label,12)
for i in range(len(nodes)-1):
    _,x,y,w,h=nodes[i]; _,x1,y1,w1,h1=nodes[i+1]; arrow(ax,(x+w+.06,y+h/2),(x1-.06,y1+h1/2))
rounded_box(ax,(3.15,.65),5.7,.65,"Regimes determine how scarce resources are spent.",12)
ax.set_title("Resource allocation flow for adaptive serving", fontsize=20, pad=12)
savefig("43_resource_allocation_flow.png")


## 3. Budget partition

Different regimes require different budget vectors. A simple normalized budget is \(B=(B_c,B_m,B_v,B_\ell,B_r)\).


In [ ]:
# 43_budget_partition.png
regimes=["balanced","compute-limited","memory-limited","latency-limited"]
categories=["compute","memory","verification","latency","reserve"]
budgets=np.array([[.26,.22,.25,.17,.10],[.18,.24,.20,.18,.20],[.28,.15,.22,.20,.15],[.25,.22,.16,.27,.10]])
fig, ax = plt.subplots(figsize=(10.5,5.8)); left=np.zeros(len(regimes)); y=np.arange(len(regimes))
for i,cat in enumerate(categories):
    ax.barh(y,budgets[:,i],left=left,label=cat)
    for j,val in enumerate(budgets[:,i]):
        if val>.14: ax.text(left[j]+val/2,j,f"{cat}\n{val:.2f}",ha="center",va="center",fontsize=9)
    left += budgets[:,i]
ax.set_yticks(y,regimes); ax.set_xlim(0,1); ax.set_xlabel("normalized resource budget")
ax.set_title("Budget partitions change by operating regime", fontsize=20)
ax.legend(ncol=5,bbox_to_anchor=(.5,-.12),loc="upper center"); ax.grid(axis="x",alpha=.25)
savefig("43_budget_partition.png")
(RESULTS/"43_budget_partition.json").write_text(json.dumps({"regimes":regimes,"categories":categories,"budgets":budgets.round(3).tolist()},indent=2),encoding="utf-8")


## 4. Regime-to-budget policy

A resource allocator maps the active regime to a budget vector: \(B_t=b(\rho_t,s_t)\).


In [ ]:
# 43_regime_to_budget_policy.png
fig, ax = plt.subplots(figsize=(12.2,5.4)); ax.axis("off"); ax.set_xlim(0,12); ax.set_ylim(0,5)
left=[("Throughput-\nbalanced",.7,3.6),("Compute-\nlimited",.7,2.55),("Memory-\nlimited",.7,1.5),("Latency-\nlimited",.7,.45)]
for label,x,y in left: rounded_box(ax,(x,y),1.75,.65,label,10)
rounded_box(ax,(4.2,2),1.85,.9,"Budget\npolicy",13)
right=[("balanced\nbudget",8.2,3.6),("reserve\ncompute",8.2,2.55),("reduce\nmemory load",8.2,1.5),("protect\nlatency",8.2,.45)]
for label,x,y in right: rounded_box(ax,(x,y),1.75,.65,label,10)
for _,x,y in left: arrow(ax,(x+1.78,y+.33),(4.15,2.45))
for _,x,y in right: arrow(ax,(6.1,2.45),(x-.08,y+.33))
ax.text(6.1,4.45,r"$B_t=b(\rho_t,s_t)$",ha="center",fontsize=16)
ax.set_title("Operating regimes map to resource-allocation policies",fontsize=20,pad=12)
savefig("43_regime_to_budget_policy.png")


## 5. Batch admission control

Resource allocation also controls which requests are admitted to the active verification batch.


In [ ]:
# 43_batch_admission_control.png
fig, ax = plt.subplots(figsize=(12.5,5.2)); ax.axis("off"); ax.set_xlim(0,12.5); ax.set_ylim(0,5)
for label,x,y,w,h in [("Incoming\nrequests",.6,3,1.55,.75),("Queue",3,3,1.35,.75),("Allocation\ncontroller",5.2,2.85,1.9,1.05),("Admitted\nbatch",8,3,1.65,.75),("Target\nverify",10.4,3,1.45,.75)]: rounded_box(ax,(x,y),w,h,label,11)
for a,b in [((2.18,3.38),(2.95,3.38)),((4.38,3.38),(5.15,3.38)),((7.15,3.38),(7.95,3.38)),((9.68,3.38),(10.35,3.38))]: arrow(ax,a,b)
for label,x,y,w,h in [("compute\nbudget",3.4,.9,1.35,.65),("memory\nbudget",5.1,.9,1.35,.65),("latency\ntarget",6.8,.9,1.35,.65)]:
    rounded_box(ax,(x,y),w,h,label,10); arrow(ax,(x+w/2,y+h+.05),(6.15,2.8))
rounded_box(ax,(4.05,4.25),4.2,.55,"Admission depends on available serving resources.",12)
ax.set_title("Batch admission control under resource constraints", fontsize=20, pad=12)
savefig("43_batch_admission_control.png")


## 6. Latency-memory tradeoff

Longer schedules can increase accepted tokens, but they also spend memory and latency headroom.


In [ ]:
# 43_latency_memory_tradeoff.png
ell=np.arange(1,9); params={"balanced":(.85,.80),"compute-limited":(.75,.72),"memory-limited":(.58,.90),"latency-limited":(.92,.55)}
fig, ax = plt.subplots(figsize=(9.5,6.2))
for name,(lat_w,mem_w) in params.items():
    latency=lat_w*(.35+.18*ell+.012*ell**2); memory=mem_w*(.45+.23*ell+.018*ell**2)
    throughput=(1-np.exp(-.55*ell))/(.6+.14*latency+.10*memory); best=np.argmax(throughput)
    ax.plot(memory,latency,marker="o",label=name); ax.scatter([memory[best]],[latency[best]],s=90); ax.text(memory[best]+.05,latency[best],f"$\\ell^*={ell[best]}$",fontsize=10)
ax.set_xlabel("memory pressure proxy"); ax.set_ylabel("latency proxy"); ax.set_title("Latency-memory tradeoff depends on operating regime", fontsize=20); ax.grid(True,alpha=.3); ax.legend()
savefig("43_latency_memory_tradeoff.png")


## 7. Allocation surface

Compute and memory budgets jointly shape throughput.


In [ ]:
# 43_allocation_surface.png
compute=np.linspace(.1,1,100); memory=np.linspace(.1,1,90); C,M=np.meshgrid(compute,memory)
throughput=(1-np.exp(-3.2*C))*(1-np.exp(-2.8*M)); throughput*=np.exp(-.45*np.maximum(C-.82,0)**2-.55*np.maximum(M-.78,0)**2); throughput*=(1-.18*np.abs(C-M))
fig, ax = plt.subplots(figsize=(9.5,6.5)); im=ax.imshow(throughput,aspect="auto",origin="lower",extent=[compute.min(),compute.max(),memory.min(),memory.max()],interpolation="bilinear")
ax.contour(C,M,throughput,levels=8,linewidths=.8,alpha=.55); ax.set_xlabel("compute budget"); ax.set_ylabel("memory budget"); ax.set_title("Resource allocation surface for adaptive serving",fontsize=20)
cbar=plt.colorbar(im,ax=ax); cbar.set_label("throughput proxy")
for label,(x,y) in {"balanced":(.62,.62),"compute-limited":(.42,.70),"memory-limited":(.72,.42),"latency reserve":(.55,.50)}.items():
    ax.scatter([x],[y],s=90); ax.text(x+.02,y+.02,label,fontsize=10)
savefig("43_allocation_surface.png")


## 8. Budget vectors

A controller can represent each regime as a normalized allocation vector.


In [ ]:
# 43_budget_vectors.png
labels=["compute","memory","verification","batch","reserve"]
vectors={"balanced":[.70,.65,.68,.62,.35],"compute-limited":[.42,.70,.50,.55,.65],"memory-limited":[.76,.38,.57,.45,.58],"latency-limited":[.62,.58,.40,.35,.78]}
x=np.arange(len(labels)); width=.18; fig,ax=plt.subplots(figsize=(10.5,5.8))
for i,(name,vals) in enumerate(vectors.items()): ax.bar(x+(i-1.5)*width,vals,width,label=name)
ax.set_xticks(x,labels); ax.set_ylim(0,1); ax.set_ylabel("normalized allocation"); ax.set_title("Regime-conditioned budget vectors",fontsize=20); ax.legend(ncol=2); ax.grid(axis="y",alpha=.3)
savefig("43_budget_vectors.png")


## 9. Adaptive allocation curves

The same additional resource budget does not help every regime equally.


In [ ]:
# 43_allocation_curves.png
budget=np.linspace(.05,1,40)
curves={"balanced":1.15*(1-np.exp(-3.8*budget)),"compute-limited":.92*(1-np.exp(-2.4*budget)),"memory-limited":1.02*(1-np.exp(-3.0*budget))-.10*np.maximum(budget-.78,0),"latency-limited":.82*(1-np.exp(-5.0*budget))-.22*np.maximum(budget-.62,0)}
fig,ax=plt.subplots(figsize=(9.5,6))
for name,vals in curves.items(): ax.plot(budget,vals,marker="o",markevery=5,label=name)
ax.set_xlabel("allocated resource budget"); ax.set_ylabel("throughput gain proxy"); ax.set_title("Adaptive throughput gains depend on regime",fontsize=20); ax.grid(True,alpha=.3); ax.legend()
savefig("43_allocation_curves.png")


## 10. Allocation phase diagram

Notebook 37 classified operating regimes. Notebook 43 classifies allocation policies.


In [ ]:
# 43_phase_diagram.png
compute_avail=np.linspace(.1,1,120); memory_avail=np.linspace(.1,1,110); C,M=np.meshgrid(compute_avail,memory_avail)
score_compute=1.2*(M-C)+.25*(1-C); score_memory=1.2*(C-M)+.25*(1-M); score_verify=.75*(C+M)-.85*np.abs(C-M)-.10; score_latency=1.05*(1-.5*(C+M))+.20*np.abs(C-M); score_balanced=1.25*np.exp(-((C-.65)/.22)**2-((M-.65)/.22)**2)
policy=np.argmax(np.stack([score_compute,score_memory,score_verify,score_latency,score_balanced]),axis=0)
fig,ax=plt.subplots(figsize=(9.5,6.5)); im=ax.imshow(policy,aspect="auto",origin="lower",extent=[compute_avail.min(),compute_avail.max(),memory_avail.min(),memory_avail.max()],interpolation="nearest")
ax.contour(C,M,policy,levels=[.5,1.5,2.5,3.5],linewidths=.9,alpha=.55); ax.set_xlabel("compute availability"); ax.set_ylabel("memory availability"); ax.set_title("Allocation-policy phase diagram",fontsize=20)
cbar=plt.colorbar(im,ax=ax,ticks=[0,1,2,3,4]); cbar.ax.set_yticklabels(["compute-heavy","memory-heavy","verification-heavy","latency-reserve","balanced"]); cbar.set_label("selected allocation policy")
for text,x,y in [("compute-\nheavy",.26,.74),("memory-\nheavy",.75,.25),("verification-\nheavy",.55,.86),("latency\nreserve",.30,.25),("balanced",.66,.64)]: ax.text(x,y,text,fontsize=11,ha="center")
savefig("43_phase_diagram.png")


## Key equations

Resource budget:

\[
B_t=(B_c,B_m,B_v,B_\ell,B_r)
\]

Regime-conditioned allocation:

\[
a_t=\pi(\rho_t,B_t,s_t)
\]

Budget constraint:

\[
\sum_i a_{t,i}\le B_t
\]

Feasible allocation set:

\[
\mathcal{A}_t
=
\{a:\; C(a)\le B_c,\;M(a)\le B_m,\;L(a)\le B_\ell\}
\]

Allocation objective:

\[
a_t^*
=
\arg\max_{a\in\mathcal{A}_t}
\Theta(a;\rho_t)
\]

Feedback update:

\[
B_{t+1}=f(B_t,\Theta_t,U_t,Q_t)
\]

where \(U_t\) is utilization and \(Q_t\) is queue state.


## Engineering summary

Notebook 43 starts the second major arc of the repository.

Operating regimes identified in Notebook 37 become resource-allocation policies. The controller spends compute, memory, verification, latency, and reserve capacity according to the active regime and the observed system state.

> **Resource allocation specifies adaptive serving.**


In [ ]:
# Save notebook manifest
manifest = {
    "notebook": "43_resource_allocation.ipynb",
    "title": "Resource Allocation",
    "engineering_statement": "Resource allocation specifies adaptive serving.",
    "figures": [
        "43_second_arc_roadmap.png",
        "43_resource_allocation_flow.png",
        "43_budget_partition.png",
        "43_regime_to_budget_policy.png",
        "43_batch_admission_control.png",
        "43_latency_memory_tradeoff.png",
        "43_allocation_surface.png",
        "43_budget_vectors.png",
        "43_allocation_curves.png",
        "43_phase_diagram.png",
    ],
    "previous_notebook": "37_operating_regimes.ipynb",
    "next_notebook": "49_adaptive_ai_infrastructure.ipynb",
}
(RESULTS / "43_resource_allocation_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
manifest


## Download artifacts

Run the final cell to package Notebook 43 outputs for download.

The archive includes:

- `43_resource_allocation.ipynb`
- generated figures
- generated JSON outputs
- notebook manifest


In [ ]:
# Package Notebook 43 artifacts for download
from pathlib import Path
from IPython.display import FileLink, display
import zipfile

CWD = Path.cwd().resolve()
if (CWD / "figures").exists() or (CWD / "results").exists() or (CWD / "notebooks").exists():
    REPO_ROOT = CWD
elif (CWD.parent / "figures").exists() or (CWD.parent / "results").exists() or (CWD.parent / "notebooks").exists():
    REPO_ROOT = CWD.parent
else:
    REPO_ROOT = CWD

NOTEBOOKS = REPO_ROOT / "notebooks"
FIGURES = REPO_ROOT / "figures"
RESULTS = REPO_ROOT / "results" / "43_resource_allocation"
SITE = REPO_ROOT / "site"
RESULTS.mkdir(parents=True, exist_ok=True)
zip_path = RESULTS / "43_resource_allocation_artifacts.zip"

notebook_candidates = [NOTEBOOKS / "43_resource_allocation.ipynb", Path.cwd() / "43_resource_allocation.ipynb"]
notebook_path = next((p for p in notebook_candidates if p.exists()), None)

paths_to_include = []
if notebook_path is not None:
    paths_to_include.append(notebook_path)
for item in [FIGURES, RESULTS, SITE]:
    if item.exists():
        paths_to_include.append(item)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for item in paths_to_include:
        item = Path(item)
        if item.is_dir():
            for path in item.rglob("*"):
                if not path.is_file() or path.resolve() == zip_path.resolve():
                    continue
                try:
                    archive_name = path.relative_to(REPO_ROOT)
                except ValueError:
                    archive_name = path.name
                zf.write(path, archive_name)
        elif item.exists():
            try:
                archive_name = item.relative_to(REPO_ROOT)
            except ValueError:
                archive_name = item.name
            zf.write(item, archive_name)

print(f"Created: {zip_path}")
print(f"Size: {zip_path.stat().st_size:,} bytes")
display(FileLink(str(zip_path)))
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass
